# 最短路径


In [ ]:
## dijkstra

# 网络延迟时间，给一个list times， 表示信号经过有向edge的传递时间，
# times[i] = (u, v, w) 表示从节点 u 到节点 v 传递信号需要时间 w。现在我们从某个节点 K 发出信号，
# 求所有节点接收到信号的最短时间，如果有节点无法接收到信号，返回 -1.





# 1 建图
# 2 确定原点， 辅助数组， dist[i]表示源点到i的最短路径，visted[i]表示i是否已经访问
# 3 初始化，dist[i] = inf, dist[s] = 0, visited[i] = False
# 4 循环，找到dist中最小的点，标记为已访问，更新dist数组，直到所有点都访问过
# 5 更新从x触到顶点y的最短路径dist[y] = min(dist[y], dist[x] + graph[x][y])
# 6 重复执行，回到第4步继续找距离最小值




inf = 1e20

# 构建一个邻接矩阵  N乘N形式
def initEges(n):
    graph = []
    for i in range(n):
        row = []
        for j in range(n):
            row.append(inf)
        graph.append(row)
    return graph


def addEdge(graph, u, v, w):

    if w < graph[u][v]:                 # 适用于有重边的dijkstra算法，取最小的边权
        graph[u][v] = w
    



def dijkstra(graph,n, s):              # 三重循环 n 是顶点
    dist = [inf for i in range(n)]                          # 这里是到n-1
    visited = [False for i in range(n)]
    dist[s] = 0
    while True:
        minDist = inf
        miIndex = 0
        for i in range(n):              # 找所有点（这个点没有访问过），将最短路径扶着到mindist中，标记这个点已访问
            if visited[i]:
                continue
            if dist[i] < minDist:
                minDist = dist[i]
                miIndex = i
        if minDist == inf:              # 所有顶点都访问过
            break
        visited[miIndex] = True

        for i in range(n):
            if visited[i]:
                continue
            w = graph[miIndex][i]                   # 边权，不能叫 dist
            if w == inf:
                continue
            if dist[miIndex] + w < dist[i]:  # 更新最短路径  (松弛操作)
                dist[i] = dist[miIndex] + w
    return dist


    
def networkDelayTime(times, n, k):
    init = initEges(n)
    for t in times:
        u = t[0] - 1                # 顶点
        v = t[1] - 1
        w = t[2]
        addEdge(init, u, v, w)
    dist = dijkstra(init, n, k - 1)         # 顶点是n， 起点是k-1
    ret = max(dist)
    if ret == inf:
        return -1
    return ret


times = [[2, 1, 1], [2, 3, 1], [3, 4, 1]]
n = 4
k = 2
print(networkDelayTime(times, n, k))


In [ ]:

# 阈值距离内邻居最少的城市


def findTheCity(n, edges, distanceThreshold):
    graph = initEges(n)
    for edge in edges:
        u = edge[0]
        v = edge[1]
        w = edge[2]
        addEdge(graph, u, v, w)
        addEdge(graph, v, u, w)  # 无向图


    reCnt = 1000000
    retIdx = -1

    for i in range(n-1, -1, -1):                     # 每个点起点求一次dijkstra最短
        dist = dijkstra(graph, n, i)            # dist是i出发到任意顶点最短路径
        cnt = 0
        for j in range(n):
            if dist[j] <= distanceThreshold:
                cnt += 1
        if cnt < reCnt:
            reCnt = cnt
            retIdx = i

    return retIdx

print(findTheCity(4, [[0, 1, 3], [1, 2, 1], [1, 3, 4], [2, 3, 1]], 4))
# Output: 3



# 前往目标的最小代价
hash ={}


def findPoint(hash, x, y):          # 每次来一个陌生的点，哈希表是会一点一点涨上去的
    if hash.get((x, y)) is None:
        hash[(x, y)] = len(hash[(x, y)])
    return hash[(x, y)]


def minicost(start, target, specialRoads):
    graph = initEges(1001)
    hash = {}
    s = findPoint(hash, start[0], start[1])
    t = findPoint(hash, target[0], target[1])
    for sr in specialRoads:
        u = findPoint(hash, sr[0], sr[1])
        v = findPoint(hash, sr[2], sr[3])
        w = sr[4]
        addEdge(graph, u, v, w)             # 特殊路径


    for (x1, y1), i in hash.items():                    # 遍历字典中的所有字段
        for (x2, y2), j in hash.items():
            if i == j:
                continue
            w = abs(x1 - x2) + abs(y1 - y2)
            addEdge(graph, i, j, w)         # 曼哈顿路径   这个的addEdge可以适用于有重边的路径，只取最小路径
    dist = dijkstra(graph, len(hash), s)
    return dist[t]                                      # dijsktra返回的是从s到所有点的最短路径，dist[t]就是从s到t的最短路径



In [17]:
# Bellman-ford 算法

# 可以求负边权， 单源最短， 顶点数X边数最好小于1000000
# edge数据结构：使用三元组来存储  -->   edge(u, v, w)   u 起点，v 终点， w 边权
# 存在负权圈的时候是最坏情况，时间复杂度是O(nm)



# A国又n个城市，很多城市的直达方式关闭， 无法通过1到达n，需要多次中转，每次中转需要进行隔离，希望你规划一条最短路线来到达城市n
# c[n-1] = 0
# a---w--->b

inf = 10000000

class Edge:
    def __init__(self, u , v, w):
        self.u = u
        self.v = v
        self.w = w
    
def doRelax(edges, dist):
    isRelax = False
    for e in edges:
        if dist[e.u] + e.w < dist[e.v]:
            dist[e.v] = dist[e.u] + e.w
            isRelax = True
    return isRelax

def bellman( n, edges, s):
    dist = [inf for i in range(n)]
    dist[s] = 0
    for i in range(n-1):
        if not doRelax(edges, dist):               # 不存在环（负权圈）
            return False, dist
    if doRelax(edges, dist):                       # 存在环（负权圈）
        return True, dist
    return False, dist



n, e = 4, 4
node = [[1,2,4], [1,3,5], [2,4,3], [3,4,5]]
c = [5, 7, 3, 4]
c[n-1] = 0
m = 0
edges = []

for i in range(e):
    u, v, w = node[i]
    m += 1
    edges.append(Edge(u-1, v-1, w + c[v-1]))
    edges.append(Edge(v-1, u-1, w + c[u-1]))

_, dist = bellman(n, edges, s=0)
print(dist)



# 英文游戏，字母阶梯, 找到从起点到终点的最短字母阶梯
# cat --> hat --> hot --> hop --> mop

# cat  hat
# hot 

# 0 --- 1
#      /
#    2



def calcEdge(a, b, l):
    ans = 0
    for i in range(l):
        if a[i] != b[i]:
            ans +=1

        if ans >1 :
            return False
    return True


strList = ["cat", "hat", "hot", "hop", "mop"]
n = len(strList)
edges = []
for i in range(n):
    for j in range(i+1, n):
        if calcEdge(strList[i], strList[j], len(strList[i])):
            edges.append(Edge(i, j, 1))
            edges.append(Edge(j, i, 1))

st = 'cat'
en = 'mop'

for i in range(n):
    if strList[i] == st:
        s = i
    if strList[i] == en:
        d = i
_, dist =  bellman(n, edges, s)

print(dist[d])



# 小松的黄牛农场
# 先求正向图中的最短路再求反图中的最短路径，然后加和一下取最大值

# 4, 8, 2
# 1, 2, 4
# 1, 3, 2
# 1, 4, 7
# 2, 1, 1
# 2, 3, 5
# 3, 1, 2
# 3, 4, 4
# 4, 2, 3

n, m, x = 4,8,2
list = [ 
        [1, 2, 4], 
        [1, 3, 2], 
        [1, 4, 7], 
        [2, 1, 1], 
        [2, 3, 5], 
        [3, 1, 2], 
        [3, 4, 4], 
        [4, 2, 3]]
u, v , w = [], [], []

for i in range(m):
    a, b, c = list[i]
    u.append(a)
    v.append(b)
    w.append(c)
edges = [None for i in range(m)]
for i in range(m):
    edges[i] = Edge(u[i]-1, v[i]-1, w[i])
dist1 = bellman(n, edges, x-1)[1]
for i in range(m):
    edges[i] = Edge(v[i]-1, u[i]-1, w[i])
dist2 = bellman(n, edges, x-1)[1]

ret = 0
for i in range(n):
    ret = max(ret, dist1[i] + dist2[i])
print(ret)

[0, 11, 8, 13]
4
10


In [ ]:
# Floyd-Warshall 算法
# 全源最短路径（定点数小于100）


# 对于n个顶点的有向图， 给定一些顶点之间的edge，e(u, v, w) 求任意两点之间的最短路径

# 方法1 枚举每个点都进行dijkstra ---> 时间复杂度是O(n3)

# 令d[k][i][j] 表示从i到j的最短路径，且路径上的所有点都在{0,1,2,...,k}中, 经过顶点是[0, k) 初始为0的时候就是不经过任何其他节点直接到j
# d[0][i][j] = graph[i][j]  # k=0时，路径上没有中间点，直接是边权


# 初始状态

# i --> j 直接有边， d[0][i][j] = graph[i][j]
# i --> j 没有边， d[0][i][j] = inf


# 状态转移

# d[k][i][j] 为只允许经过节点[0. k)作为中间节点的情况， i到j的最短路径
# 如果最短路径不经过k点， 则d[k][i][j] = d[k-1][i][j]
# 如果最短路径经过k点    则d[k][i][j] = d[k-1][i][k] + d[k-1][k][j]

# 状态转移方程：d[k][i][j] = min(d[k-1][i][j], d[k-1][i][k] + d[k-1][k][j])
# 空间优化    d[i][j] = min(d[i][j], d[i][k] + d[k][j])  只需要二维数组就可以了

# 邻接矩阵只要顶点数小于1000就没问题



# 题目
# 有n个网络节点标记为1--->n， 给一个列表times表示信号经过有向边的传递时间
# times[1] = [u1, v1, w1]，其中u1源节点， v1目标节点， w1信号从源节点到目标节点花费时间。
# 如果某个节点k发出一个信号， 需要多久才能使所有节点都受到信号？如果不能使所有节点受到信号返回 -1 


# 初始化邻接矩阵
inf = 1e20
def initEdges(n):               #
    f = [[] for i in range(n)]
    for i in range(n):
        for j in range(n):
            f[i].append(0 if i == j else inf)
    return f
# 边添加
def addEdge(f, u, v, w):        # f 是距离矩阵
    if w < f[u][v]:                
        f[u][v] = w

# 建图
# def buildGraph(f, times):
#     for time in times:
#         u, v, w = time
#         addEdge(f, u - 1, v - 1, w)  # 转换为0索引

# 框架代码， k循环， i循环， j循环
def FloyedWarshall(f, n):
    for k in range(n):
        for i in range(n):
            for j in range(n):

                if f[i][k] == inf or f[k][j] == inf:
                    continue
                if i == k or j == k:
                    continue
                if f[i][k] + f[k][j] < f[i][j]:
                    f[i][j] = f[i][k] + f[k][j]

def networkDelayTime(times, n, k):
    f = initEdges(n)
    for time in times:
        u, v, w = time
        addEdge(f, u - 1, v - 1, w)  # 转换为0索引
    FloyedWarshall(f, n)
    ret = max(f[k - 1])                                 # k-1 就是到终点的所有点？？？？？
    return -1 if ret == inf else ret






In [ ]:


# 到达目的地的方案数


mod = 100000
inf = 1e20
def initEdges(n):             
    f = [[] for i in range(n)]
    cnt = [[] for i in range(n)]            # 某个点到另一个点的最短路方案数
    for i in range(n):
        for j in range(n):
            f[i].append(0 if i == j else inf)
            cnt[i].append(1 if i == j else 0)  # 初始化方案数，自己到自己有1种方案
    return f, cnt

# 边添加
def addEdge(f,cnt,  u, v, w):        # f 是距离矩阵
    if w < f[u][v]:                
        f[u][v] = w
    if w == f[u][v]:                         # 加一边方案数加1
        cnt[u][v] += 1

# 框架代码， k循环， i循环， j循环
def FloyedWarshall(f, cnt, n):
    for k in range(n):
        for i in range(n):
            for j in range(n):

                if f[i][k] == inf or f[k][j] == inf:
                    continue
                if i == k or j == k:
                    continue
                if f[i][k] + f[k][j] < f[i][j]:
                    f[i][j] = f[i][k] + f[k][j]
                    cnt[i][j] = cnt[i][k] * cnt[k][j]%mod
                elif f[i][k] + f[k][j] == f[i][j]:              # 如果长度一样的话
                    cnt[i][j] += cnt[i][k] * cnt[k][j]%mod
                    if cnt[i][j] >= mod:
                        cnt[i][j] %=mod

def countPaths(n, roads):
    f, cnt = initEdges(n)
    for road in roads:
        u, v, w = road
        addEdge(f, cnt, u - 1, v - 1, w)  # 转换为0索引
        addEdge(f, cnt, v - 1, u - 1, w)  # 无向图
    FloyedWarshall(f, cnt, n)
    return cnt[0][n - 1] 


print(countPaths(7, [[0,6,7],[0,1,2],[1,2,3],[1,3,3],[6,3,3],[3,5,1],[6,5,1],[2,5,1],[0,4,5],[4,6,2]]))

1


In [24]:
# dijkstra优化算法， dijkstra + 堆数据结构， 时间复杂是O((m + n)logn)

# 只能处理非负边权
# 分析单源最短路
# 顶点数量< 1000000， 边数量< 1000000
# 数据结构，邻接表，哈希表，二叉堆



# 建图

# 辅助数组
# 对于图G(V, E)  V是顶点集合， E是边集合, 源点是s，dist[i]表示s到i的最短路径， visited[i]表示i是否已经访问，即最短路径是否确定

# 辅助堆
# 利用一个小顶堆heap存放二元组(v, dist[v]), 小顶堆是有向对列的作用， dist[v]值越小，会从优先对列中先出队

# 堆的作用，对于朴素dijkstra算法，获取距离要用遍历的方式，使用堆之后就可以O(logn)的时间复杂度获取最小距离的点

# 初始化数据
# dist[i] = inf
# visited[i] = False
# dist[s] = 0
# heap.push((s, dist[s]))


# 找距离最小的点
# dist[v] = min(dist[v], dist[u] + w[u][v])  # 松弛操作




import heapq
inf = 1000000000

# 定义距离二元组
class Dist:
    def __init__(self, v, w):
        self.v = v
        self.w = w
    def __lt__(self, other):
        return self.w < other.w
# 初始化邻接表
def initEdges(n):
    return [[] for _ in range(n)]

# 邻接表加边

def addEdge(edges, u, v, w):
    edges[u].append(Dist(v, w))  # 有向图

# 建图


# 框架代码``

def dijkstraInit(n, st, heap, visited, d):
    for i in range(0, n):
        d[i] = inf
        visited[i] = False
    d[st] = 0
    heapq.heappush(heap, Dist(st, d[st]))


def dijkstraFindMin(heap):
    return heapq.heappop(heap).v


def dijkstraUpdate(u, edges, heap, visited, d):
    if visited[u]:
        return
    visited[u] = True
    for edge in edges[u]:
        v = edge.v
        w = edge.w
        if d[u] + w < d[v]:
            d[v] = d[u] + w
            heapq.heappush(heap, Dist(v, d[v]))


def DijkstraHeap(n, st, edges):
    heap = []
    visited = [False for _ in range(n)]
    d = [0 for _ in range(n)]
    dijkstraInit(n, st, heap, visited, d)
    while len(heap) > 0:
        u = dijkstraFindMin(heap)
        dijkstraUpdate(u, edges, heap, visited, d)
    return d



n, m = 5, 8 
edges = [[1, 4, 5], [2, 4, 7], [3, 5, 8], [1, 3, 8], [5, 1, 12], [4, 2, 7], [4, 5, 10], [2, 5, 3]]
init_edges = initEdges(n)
for i in range(m):
    u, v, w = edges[i]
    addEdge(init_edges, u - 1, v - 1, w)  # 转换为0索引
d = DijkstraHeap(n, 0, init_edges)
ans = d[n - 1]
if ans == inf:
    print(-1)
else:
    print(ans)



# 蓝桥王国
n, m = 3, 3
edges = [[1, 2, 1], [2, 3, 1], [1, 3, 3]]
init_edges = initEdges(n)
for i in range(m):
    u, v, w = edges[i]
    addEdge(init_edges, u - 1, v - 1, w)  # 转换为0索引
d = DijkstraHeap(n, 0, init_edges)
for x in d:
    if x == inf:
        print(-1)
    else:
        print(x)

    

15
0
1
2


In [ ]:
# 最短路径SPFA 


